In [6]:
import requests

In [7]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [8]:
len(zip_response.content)

17544837

In [9]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [30]:
file_names =zip_archive.namelist()
file_names[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [49]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_file_content = mdx_file.read().decode('utf8')

In [45]:
print(mdx_file_content)

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.
</Check>

![](/images/alerts.png)

To enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:

* A notification channel.

* An alert condition.

## Notification channels

You can choose between the following options:

* **Email**. Add email addresses to send alerts to.

* **Slack**. Add a Slack webhook.

* **Discord**. Add a Discord webhook.

## Alert conditions

### Failed tests

If you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.

<Tip>
  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if th

In [48]:
import frontmatter

post = frontmatter.loads(mdx_file_content)

In [34]:
post.metadata

{}

In [35]:
print(post.content[:100])

mdx_file_content


In [36]:
_, filename_corrected = filename.split('/', maxsplit=1)
print(filename_corrected)

docs/platform/alerts.mdx


In [52]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [53]:
documents = []
with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith(('md', 'mdx')):
            continue
        with zip_ref.open(file_path) as file:
             content = file.read().decode('utf-8')
             post = frontmatter.loads(content)
             doc = {
                'content': post.content,
                'title': post.metadata.get('title'),
                'description': post.metadata.get('description'),
                "filename": file_path.split('/', 1)[-1]
             }
             documents.append(doc)

In [54]:
len(documents)

95

In [55]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},    
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [58]:
md_file = files[10]

In [56]:
documents = [f.parse() for f in files]

In [59]:
len(documents)

95

In [60]:
documents[10] 

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

### Search

In [61]:
query = 'LLM as a Judge'

rag:

1.search<---

2.prompt

3.llm

In [62]:
from minsearch import Index

In [63]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [64]:
results = index.search(query, num_results=5)

In [65]:
len(results)

5

In [67]:
len(results[0]['content'])

21834

### Chunking

In [68]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


rag: 

1. search <-- 5 docs
2. prompt <-- 5 x 20k = 100k
3. llm

In [70]:
document = list(range(0,100))

In [71]:
window_size = 10
start = 0
step = 5

chunks = []

while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [72]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break
    
    return chunks

In [73]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [74]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [75]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [76]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [77]:
results = chunk_index.search(query)

In [78]:
from gitsource import chunk_documents

In [79]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

### RAG

In [80]:
from openai import OpenAI

openai_client = OpenAI()

In [81]:
search_result = chunk_index.search(query, num_results=5)

In [82]:
query = 'how do I implement llm as a judge?'

In [83]:
import json

In [84]:
search_result_json = json.dumps(search_result, indent=2)

In [85]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()

True

In [ ]:
# import os
# from openai import OpenAI

# client = OpenAI(
#     api_key=os.getenv("CLOUDFLARE_API_TOKEN"),
#     base_url=f"https://api.cloudflare.com/client/v4/accounts/{os.getenv('CLOUDFLARE_ACCOUNT_ID')}/ai/v1"
# )

In [101]:
def llm(user_prompt, instructions=None, model="openai/gpt-oss-120b:free"):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.choices[0].message.content

In [102]:
answer = llm(user_prompt, instructions)

In [103]:
print(answer)

Below is a quick‑start guide that shows **how to turn an LLM (e.g. OpenAI’s GPT‑4o‑mini) into a “judge”** that scores your model’s answers.  
The code follows the official Evidently AI tutorial, so you can copy‑paste it into a Jupyter notebook or a Colab notebook and have a working LLM‑judge in a few minutes.

---

## 1️⃣ Install the library & set your API key  

```python
!pip install evidently   # ← runs once

import os
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"
```

---

## 2️⃣ Create a tiny evaluation data‑set  

You need at least three columns:

| `question` | `target_response` (ground‑truth) | `new_response` (model output) |
|------------|----------------------------------|-------------------------------|

```python
import pandas as pd

data = [
    [
        "Hi there, how do I reset my password?",
        "To reset your password, click on 'Forgot Password' on the login page and follow the instructions sent to your registered email.",
        "To change your password, sele

In [104]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [105]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_result, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [106]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [108]:
rag('how do I implement llm as a judge?')

'Below is a concise, step‑by‑step recipe that shows how you can turn an LLM (e.g., OpenAI\u202fGPT‑4o‑mini) into an **automatic judge** for your own text‑generation tasks.  \nThe code is based on the Evidently\u202fAI library, which already provides ready‑made prompt templates and utilities for scoring, reporting, and (optionally) pushing the results to Evidently\u202fCloud.\n\n---\n\n## 1️⃣  Prerequisites  \n\n| What you need | Why |\n|---------------|-----|\n| **Python 3.8+** | Run the notebooks / scripts |\n| **OpenAI (or another) API key** | The LLM that does the judging |\n| **`evidently` package** (`pip install evidently`) | Provides `LLMEval`, prompt templates, dataset helpers, and reports |\n| (Optional) **Jupyter / Colab** | Easy visualisation of the HTML reports |\n\n---\n\n## 2️⃣  Install & import\n\n```python\npip install evidently   # run once\n\nimport os\nimport pandas as pd\nfrom evidently import Dataset, DataDefinition, Report\nfrom evidently.presets import TextEvals, 